In [0]:
%sql
create catalog if not exists workforce_analytics;
create schema if not exists workforce_analytics.bronze;
create volume if not exists workforce_analytics.bronze.raw_data

In [0]:
import requests
import os
import pyspark.sql.functions  as F
github_owner = "rajupatel07"
repo_name = "workforce_analytics-7trent"
branch = "main"
# GitHub folder containing your source data
github_source_path = "source"
volume_path = "/Volumes/workforce_analytics/bronze/raw_data"
# 3. GitHub API URL
api_url = (
    f"https://api.github.com/repos/"
    f"{github_owner}/{repo_name}/git/trees/{branch}?recursive=1"
)

# Get the list of files in the repository
response = requests.get(api_url)
response.raise_for_status()

repo_files = response.json()["tree"]

# --------------------------------------------------
# 4. Download files from GitHub
# --------------------------------------------------

for file in repo_files:

    # Only download files inside source/
    if (
        file["type"] == "blob"
        and file["path"].startswith(github_source_path + "/")
        and file["path"].lower().endswith(".csv")
    ):

        # Keep the original folder structure
        relative_path = file["path"]

        # Example:
        # source/facts/fact_work-01_04_2026.csv

        destination_path = os.path.join(
            volume_path,
            relative_path
        )

        # Create the required folders
        os.makedirs(
            os.path.dirname(destination_path),
            exist_ok=True
        )

        # Create the raw GitHub download URL
        download_url = (
            f"https://raw.githubusercontent.com/"
            f"{github_owner}/{repo_name}/"
            f"{branch}/{relative_path}"
        )

        # Download the CSV file
        file_response = requests.get(download_url)
        file_response.raise_for_status()

        # Save the file into the Volume
        with open(destination_path, "wb") as f:
            f.write(file_response.content)

        print(f"Downloaded: {relative_path}")

print("All CSV files downloaded successfully!")

In [0]:
df=spark.read.csv("/Volumes/workforce_analytics/bronze/raw_data/source/dimensions/DimCancellationReason.csv",header=True,inferSchema=True)
display(df)